# Day 09. Exercise 03
# Ensembles

## 0. Imports

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, BaggingClassifier, StackingClassifier
from sklearn.metrics import precision_score, recall_score
import joblib

## 1. Preprocessing

1. Create the same dataframe as in the previous exercise.
2. Using `train_test_split` with parameters `test_size=0.2`, `random_state=21` get `X_train`, `y_train`, `X_test`, `y_test` and then get `X_train`, `y_train`, `X_valid`, `y_valid` from the previous `X_train`, `y_train`. Use the additional parameter `stratify`.

In [3]:
df = pd.read_csv('../data/day-of-week-not-scaled.csv')
df_scaled = pd.read_csv('../data/dayofweek.csv')
df['dayofweek'] = df_scaled['dayofweek']

In [4]:
y = df.dayofweek
X = df.drop('dayofweek', axis=1)


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=21, stratify=y)

In [5]:
X_train, X_valid, y_train, y_valid = train_test_split(X_train, y_train, test_size=0.2, random_state=21, stratify=y_train)

## 2. Individual classifiers

1. Train SVM, decision tree and random forest again with the best parameters that you got from the 01 exercise with `random_state=21` for all of them.
2. Evaluate `accuracy`, `precision`, and `recall` for them on the validation set.
3. The result of each cell of the section should look like this:

```
accuracy is 0.87778
precision is 0.88162
recall is 0.87778
```

In [6]:
def print_metrics(model, X_valid, y_valid):
    y_pred = model.predict(X_valid)

    print(f'accuracy is {model.score(X_valid, y_valid):.5f}')
    print(f'precision is {precision_score(y_valid, y_pred, average="weighted"):.5f}')
    print(f'recall is {recall_score(y_valid, y_pred, average="weighted"):.5f}')

In [7]:
svm = SVC(C=10, gamma='auto', random_state=21, probability=True)
svm.fit(X_train, y_train)

print_metrics(svm, X_valid, y_valid)

accuracy is 0.87778
precision is 0.88162
recall is 0.87778


In [8]:
dt = DecisionTreeClassifier(class_weight='balanced', criterion='gini', max_depth=21, random_state=21)
dt.fit(X_train, y_train)

print_metrics(dt, X_valid, y_valid)

accuracy is 0.86667
precision is 0.87170
recall is 0.86667


In [9]:
rf = RandomForestClassifier(class_weight='balanced', criterion='entropy', max_depth=24, n_estimators=100, random_state=21)
rf.fit(X_train, y_train)

print_metrics(rf, X_valid, y_valid)

accuracy is 0.89630
precision is 0.89698
recall is 0.89630


## 3. Voting classifiers

1. Using `VotingClassifier` and the three models that you have just trained, calculate the `accuracy`, `precision`, and `recall` on the validation set.
2. Play with the other parameteres.
3. Calculate the `accuracy`, `precision` and `recall` on the test set for the model with the best weights in terms of accuracy (if there are several of them with equal values, choose the one with the higher precision).

In [10]:
vclf = VotingClassifier(estimators=[
    ('svm', svm), 
    ('dt', dt), 
    ('rf', rf)
], voting='hard')

vclf = vclf.fit(X_train, y_train)

In [11]:
print_metrics(vclf, X_valid, y_valid)

accuracy is 0.90000
precision is 0.89993
recall is 0.90000


In [12]:
vclf2 = VotingClassifier(estimators=[
    ('svm', svm), 
    ('dt', dt), 
    ('rf', rf)
], voting='soft')

vclf2 = vclf2.fit(X_train, y_train)

print('VotingClassifier with param voting=soft\n')
print_metrics(vclf2, X_valid, y_valid)

VotingClassifier with param voting=soft

accuracy is 0.88519
precision is 0.88840
recall is 0.88519


In [22]:
vclf3 = VotingClassifier(estimators=[
    ('svm', svm), 
    ('dt', dt), 
    ('rf', rf)
], voting='hard', weights=[2, 3, 2])

vclf3 = vclf3.fit(X_train, y_train)

print_metrics(vclf3, X_valid, y_valid)

accuracy is 0.90741
precision is 0.90773
recall is 0.90741


In [21]:
vclf4 = VotingClassifier(estimators=[
    ('svm', svm), 
    ('dt', dt), 
    ('rf', rf)
], voting='soft', weights=[4, 1, 4])

vclf4 = vclf4.fit(X_train, y_train)

print_metrics(vclf4, X_valid, y_valid)

accuracy is 0.91111
precision is 0.91288
recall is 0.91111


In [15]:
print_metrics(vclf4, X_test, y_test)

accuracy is 0.91420
precision is 0.91713
recall is 0.91420


## 4. Bagging classifiers

1. Using `BaggingClassifier` and `SVM` with the best parameters create an ensemble, try different values of the `n_estimators`, use `random_state=21`.
2. Play with the other parameters.
3. Calculate the `accuracy`, `precision`, and `recall` for the model with the best parameters (in terms of accuracy) on the test set (if there are several of them with equal values, choose the one with the higher precision)

In [16]:
svm = SVC(C=10, gamma='auto', random_state=21, probability=True)
bclf = BaggingClassifier(base_estimator=svm, n_estimators=10, random_state=21).fit(X_train, y_train)

In [17]:
print_metrics(bclf, X_valid, y_valid)

accuracy is 0.88519
precision is 0.89427
recall is 0.88519


In [18]:
param_grid = {
    'n_estimators': [20, 30, 50],
    'max_samples': [0.5, 0.7, 1.0],
    'max_features': [0.5, 0.7, 1.0]
}

In [19]:
bclf_grid_search = GridSearchCV(BaggingClassifier(base_estimator=svm, random_state=21), param_grid, cv=5)

In [20]:
bclf_grid_search.fit(X_train, y_train)


KeyboardInterrupt



In [ ]:
bclf_cv_df = pd.DataFrame(bclf_grid_search.cv_results_)

bclf_scores = bclf_cv_df[['param_n_estimators', 'param_max_samples', 'param_max_features', 'mean_test_score', 'rank_test_score']].sort_values('rank_test_score')

bclf_scores.head()

In [ ]:
bclf = BaggingClassifier(base_estimator=svm, n_estimators=50, random_state=21).fit(X_train, y_train)
print_metrics(bclf, X_test, y_test)

## 5. Stacking classifiers

1. To achieve reproducibility in this case you will have to create an object of cross-validation generator: `StratifiedKFold(n_splits=n, shuffle=True, random_state=21)`, where `n` you will try to optimize (the details are below).
2. Using `StackingClassifier` and the three models that you have recently trained, calculate the `accuracy`, `precision` and `recall` on the validation set, try different values of `n_splits` `[2, 3, 4, 5, 6, 7]` in the cross-validation generator and parameter `passthrough` in the classifier itself,
3. Calculate the `accuracy`, `precision`, and `recall` for the model with the best parameters (in terms of accuracy) on the test set (if there are several of them with equal values, choose the one with the higher precision). Use `final_estimator=LogisticRegression(solver='liblinear')`.

In [23]:
n_splits = [2, 3, 4, 5, 6, 7]
passthrough = [True, False]

estimators=[
    ('svm', svm), 
    ('dt', dt), 
    ('rf', rf)
]

for n in n_splits:
    for p in passthrough:
        kfold = StratifiedKFold(n_splits=n, shuffle=True, random_state=21)
        clf = StackingClassifier(
            estimators=estimators, 
            cv=kfold, 
            passthrough=p,
            final_estimator=LogisticRegression(solver='liblinear')
        )
        clf.fit(X_train, y_train)
        print(f'n: {n}, p: {p}')
        print_metrics(clf, X_valid, y_valid)

n: 2, p: True
accuracy is 0.90370
precision is 0.90619
recall is 0.90370
n: 2, p: False
accuracy is 0.89630
precision is 0.89678
recall is 0.89630
n: 3, p: True
accuracy is 0.90370
precision is 0.90632
recall is 0.90370
n: 3, p: False
accuracy is 0.89630
precision is 0.89759
recall is 0.89630
n: 4, p: True
accuracy is 0.91111
precision is 0.91327
recall is 0.91111
n: 4, p: False
accuracy is 0.90370
precision is 0.90570
recall is 0.90370
n: 5, p: True
accuracy is 0.90000
precision is 0.90217
recall is 0.90000
n: 5, p: False
accuracy is 0.90000
precision is 0.90056
recall is 0.90000
n: 6, p: True
accuracy is 0.90370
precision is 0.90450
recall is 0.90370
n: 6, p: False
accuracy is 0.90370
precision is 0.90436
recall is 0.90370
n: 7, p: True
accuracy is 0.90370
precision is 0.90640
recall is 0.90370
n: 7, p: False
accuracy is 0.90370
precision is 0.90538
recall is 0.90370


In [24]:
kfold = StratifiedKFold(n_splits=4, shuffle=True, random_state=21)
clf = StackingClassifier(
    estimators=estimators, 
    cv=kfold, 
    passthrough=True,
    final_estimator=LogisticRegression(solver='liblinear')
)

In [25]:
clf.fit(X_train, y_train)
print_metrics(clf, X_test, y_test)

accuracy is 0.90533
precision is 0.90844
recall is 0.90533


## 6. Predictions

1. Choose the best model in terms of accuracy (if there are several of them with equal values, choose the one with the higher precision).
2. Analyze: for which weekday your model makes the most errors (in % of the total number of samples of that class in your full dataset), for which labname and for which users.
3. Save the model.

In [26]:
vclf4 = VotingClassifier(estimators=[
    ('svm', svm), 
    ('dt', dt), 
    ('rf', rf)
], voting='soft', weights=[4, 1, 4])

vclf4 = vclf4.fit(X_train, y_train)
y_pred = vclf4.predict(X_test)

In [27]:
errors_df = pd.DataFrame({
    'actual': y_test
})

ohe_labname_cols = [c for c in X_test.columns if c.startswith('labname')]
ohe_uid_cols = [c for c in X_test.columns if c.startswith('uid')]

errors_df['labname'] = (
    X_test[ohe_labname_cols]
    .idxmax(axis=1)
    .str.replace('labname_', '')
)

errors_df['uid'] = (
    X_test[ohe_uid_cols]
    .idxmax(axis=1)
    .str.replace('uid_', '')
)


errors_df['is_error'] = y_test != y_pred

errors_df

,actual,labname,uid,is_error
1087,1,project1,user_14,False
16,5,laba04s,user_2,False
563,6,laba05,user_12,False
1381,3,project1,user_20,False
1199,2,project1,user_28,False
...,...,...,...,...
1411,3,project1,user_4,False
1079,1,project1,user_14,False
1222,2,project1,user_29,False
1064,1,project1,user_14,False


In [28]:
weekday_errors = errors_df.groupby('actual').is_error.agg(
    count='size',
    errors_count='sum'
)

weekday_errors['error %'] = np.round(weekday_errors['errors_count'] / weekday_errors['count'] * 100, 2)

weekday_errors.sort_values('error %', ascending=False)

,count,errors_count,error %
actual,,,
0,27,8,29.63
5,54,7,12.96
4,21,2,9.52
6,71,6,8.45
1,55,4,7.27
2,30,2,6.67
3,80,3,3.75


In [29]:
lab_errors = errors_df.groupby('labname').is_error.agg(
    count='size',
    errors_count='sum'
)

lab_errors['error %'] = np.round(lab_errors['errors_count'] / lab_errors['count'] * 100, 2)

lab_errors.sort_values('error %', ascending=False).head()

,count,errors_count,error %
labname,,,
lab03,1,1,100.00
laba04,35,9,25.71
laba04s,25,6,24.00
lab05s,6,1,16.67
laba06,9,1,11.11


In [30]:
uid_errors = errors_df.groupby('uid').is_error.agg(
    count='size',
    errors_count='sum'
)

uid_errors['error %'] = np.round(uid_errors['errors_count'] / uid_errors['count'] * 100, 2)

uid_errors.sort_values('error %', ascending=False).head()

,count,errors_count,error %
uid,,,
user_6,4,2,50.00
user_17,7,2,28.57
user_3,14,3,21.43
user_16,5,1,20.00
user_18,6,1,16.67


In [31]:
joblib.dump(vclf4, '../data/vclf4_model.joblib')

['../data/vclf4_model.joblib']